##Live Lesson Notebook: Week 10 - Chains & Tools

In this notebook we will play with agents and chains implemented through LangChain. Note that you will not use a GPU for this notebook, a CPU is sufficient.

We will first do the usual installs and imports:

In [1]:
%%capture

!pip install -q -U langchain
!pip install -q -U langchain-core
!pip install -q -U langchain-classic
!pip install --upgrade --quiet  langchain-community
!pip install -q -U langchainhub
!pip install -q -U langchain-openai
!pip install pydantic
!pip install -U -q duckduckgo_search
!pip install -U -q ddgs


In [2]:
import torch
import os
import pprint
import langchain_classic
#langchain.__version__
import langchain_community

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.output_parsers import StrOutputParser

from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_classic import hub
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.document_loaders import TextLoader


from langchain_core.runnables import RunnablePassthrough
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_community.llms import OpenAI
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# Import things that are needed generically
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool, StructuredTool, tool
from langchain_community.tools import DuckDuckGoSearchRun, DuckDuckGoSearchResults


from langchain.agents import create_agent
from langchain_core.tools import tool

from google.colab import userdata

/tmp/ipykernel_19134/3261104164.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


###1. Chains and Chains within Chains

Now we will build our first chain. We will largely use OpenAI's GPT-3.5 Instruct model for this purpose, as it is substantially cheaper than GPT-4, and Agentic workflows can incur substantial LLM usage (be aware!). (We will however in the end also use GPT-4o to compare Agent behaviors.)  

 In order to use GPT, you have to use your OpenAI API key. (**Do NOT** print it out in the notebook! Keep it in the secrets on the left and import it as we do below, but do not show it in the notebook as clear text.)

In [3]:
OPEN_AI_KEY = userdata.get('OPENAI_API_KEY')

model = OpenAI(openai_api_key=OPEN_AI_KEY, model="gpt-3.5-turbo-instruct")
model_gpt_4o = ChatOpenAI(openai_api_key=OPEN_AI_KEY, model="gpt-4o-mini")

/tmp/ipykernel_19134/1937180582.py:3: LangChainDeprecationWarning: The class `OpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import OpenAI``.
  model = OpenAI(openai_api_key=OPEN_AI_KEY, model="gpt-3.5-turbo-instruct")


Let us create the first Chain. At a minimum, we need a prompt template and an LLM. An output parsers is also useful to have:

In [4]:
prompt1 = ChatPromptTemplate.from_template("In which city was {person} born? Give me only the city! Do not say '<person> was born in <city>', but just '<city>' ")

chain1 = prompt1 | model | StrOutputParser()

How do we run the chain? Let's select 'John Lennon' as the person.

In [5]:
output_1 = chain1.invoke({"person": "John Lennon"})
output_1

'\n\nLiverpool'

Perfect! What if you want a second step to get the state of the city we got in step 1? And also get the output in another language? Can we 1) re-use chain 1, and 2) pass a second parameter for this next step? We can! Here it is:

In [6]:
prompt2 = ChatPromptTemplate.from_template(
    "Give me the country in which the city {city} is located. And also give me this attribute about that country: {country_attribute}"
)
chain2 = (
    {"city": chain1, "country_attribute": itemgetter("country_attribute")}
    | prompt2
    | model

)


In [7]:
output_2 = chain2.invoke({"person": "John Lennon", "country_attribute": "founding date"}
              )
print(output_2)



The country in which Liverpool is located is England. The founding date of England is traditionally considered to be 927 AD, when the Kingdom of England was established. However, the exact founding date is debated and some historians suggest it was actually earlier. 


Great. The first Chain produced 'Augsburg', which then in turn became one of the inputs of the second Chain, which produced the country and the requested attribute.


##2. Tools & API Calls

Now we will see how we can call a DuckDuckGo Search within the Chain. This is an example of the API calls that we discussed.

First, what does the API look like?

In [8]:
search = DuckDuckGoSearchRun()

search.run("George Washington birthdate")

"George Washington's Early Years. 2. An Officer and Gentleman Farmer. 3. George Washington During the American Revolution. 4. America’s First President. Birthday of George Washington | February 22,1732 #georgewashington #georgewashingtonhair #usapresident #ushistory #uspresident #uspresidents #washington. Discover the Home of George and Martha Washington. Open 365 days a year, Mount Vernon is located just 15 miles south of Washington DC. Explore George Washington birth charts with insights into Sun, Moon, Rising signs, zodiac patterns, and compatibility. Learn what the stars say about your favorite famous figures. George Washington lived at Mount Vernon for more than 40 years. The big wooden house is 24 kilometers south of Washington, DC -- the city named in his honor."

Note that this is a search result and not an LLM answer! So you recover text which usually contains a lot more information than simply the answer to your question. (That's one reason why RAG is superior to search.)

Now we will put it in a chain. We will do this in three steps:

1) rewrite the question as a search query using the LLM.     
2) Send the query to the search tool.   
3) construct the answer using the LLM.




In [9]:
template_search_rewrite = """Turn the following user input into a search query for a search engine:\n\n
{input}"""

template_answer = """Based on this search result:\n\n
{search_result},
\n\n
think through it step by step to give an answer to this purpose: {purpose}.
End your answer with:
Final answer: <just the answer addressing the purpose, not more>
"""

prompt_search_rewrite = ChatPromptTemplate.from_template(template_search_rewrite)

prompt_answer = ChatPromptTemplate.from_template(template_answer)

search = DuckDuckGoSearchRun()


chain_search_query_rewrite = prompt_search_rewrite | model | StrOutputParser()

chain_search = chain_search_query_rewrite | search

full_chain = (
    {"search_result": chain_search, "input": itemgetter("input"), "purpose": itemgetter("purpose")}
    | prompt_answer
    | model
    | StrOutputParser()
)


As we can see, *chain_search_query* using the LLM rewrites the 'input' (a thought) into a more suitabke search query. *chain_search* inherits that chain and adds a tool use, and *full_chain* combines the search result and the 'purpose' to construct the answer using the LLM.

Let's see the outputs of all of the (sub-)chains for this situation:

* input:   *I wonder when George Washington was born.*
* purpose: *Washington's age in 1767*

Note that the *chain_search_query* and *chain_search* chains do not depend on the 'purpose', so we will not provide that argument for theior invocations.

In [10]:
chain_search_query_rewrite.invoke({"input": "I wonder when George Washington was born."})

'\n\n"George Washington birth date"'

In [11]:
chain_search.invoke({"input": "I wonder when George Washington was born."})

"16 Feb 2026 ... ... George Washington's birthdate. George Washinton February 22, 1732 to December 14, 1799. George Washington was commander in chief of the Continental Army ... 22 Feb 2026 ... George Washington's birthdate difference between Julian and Gregorian calendars. Wayne County, NY - Historic to the Core ▻ Huron, New York. 2y · Public · Did ... 15 Jun 2026 ... partially verifiable George Washington's birthdate, although not definitively fixed in contemporary records, reflects a historical practice of less precise ... 4 Feb 2026 ... ... George Washington's birthdate. George Washinton February 22, 1732 to December 14, 1799. George Washington was commander in chief of the Continental Army ... 5 days ago ... George Washington's birthdate, although not definitively fixed in contemporary records, reflects a historical practice of less precise record-keeping."

In [12]:
print(full_chain.invoke({"input": "I wonder when George Washington was born.",
                   "purpose": "Washington's age in 1767"}))


1. Determine the year of 1767.
1767 is a year in the mid-18th century, specifically between 1761 and 1770.

2. Calculate the difference between 1767 and Washington's birth year.
1767 - 1732 = 35

3. Add the difference to Washington's birth year to find his age.
1732 + 35 = 1767

4. Final answer: In 1767, George Washington was 35 years old.


Good. This shows the use of API calls together with LLMs.

We will now turn to agents using the new LangGraph.